# Introduction

This notebook showcases the possibility to provide custom clusters indead of relying on the provided clustering mechanism.

# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from nedis.parallelization import set_threads_for_external_libraries
set_threads_for_external_libraries(4)

In [ ]:
import sys
import logging
import pathlib

import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns

import scipy
from sklearn.manifold import TSNE

In [ ]:
import nedis.data.synthetic
import nedis.visualization
import nedis.cluster.leidenalg

from nedis.cordis.disruption \
    import CorrelationDisruption \
    as CorrelationDisruptionTransformer
from nedis.data.synthetic import make_correlation_data_mixed
from nedis.visualization import plot_cordis_cluster

In [ ]:
from nedis.visualization import plot_cordis_cluster as plot_cluster

In [ ]:
from nedis.cordis.utils import calculate_disruption_values_for_cluster

In [ ]:
from nedis.base import calculate_correlation_disruption_matrix_cv

In [ ]:
logging.basicConfig(stream=sys.stdout)
logging.getLogger("nedis").setLevel(level=logging.DEBUG)

# Config

In [ ]:
# randomness
random_state = 43
np.random.seed(random_state)

In [ ]:
output_dir = pathlib.Path("../_out/synthetic")
output_dir.mkdir(parents=True, exist_ok=True)

# Data

In [ ]:
n_samples = 100
correlations = np.linspace(0.1,.9, 5)
data = [
    make_correlation_data_mixed(
        [5,10,5,5], 
        correlations=np.array([
            [1-c,0,0,0],
            [0,c,0,0],
            [0,0,1-c,-(1-c)],
            [0,0,-(1-c),1-c]]), 
        n_noise_features=15, 
        n_samples=n_samples,
        random_state=random_state + i) 
    for i, c in enumerate(correlations)]

X = np.concatenate(data)
y = np.concatenate([np.repeat(i, d.shape[0]) for i, d in enumerate(data)])
entities = np.tile(np.arange(n_samples), len(correlations))
labels = np.repeat([0,1,2,-1], [5,10,10,15])

In [ ]:
y_unique = np.unique(y)
correlation_matrices = {}
for yy in y_unique: 
    cor = np.corrcoef(X[y == yy,:], rowvar=False)
    correlation_matrices[yy] = cor

In [ ]:
coordinates = {}
for yy, correlation_matrix in correlation_matrices.items(): 
    tsne = TSNE(
        n_components=2, random_state=random_state, init="pca", learning_rate=200)
    coo = tsne.fit_transform(abs(correlation_matrix))
    coordinates[yy] = coo

In [ ]:
fig, axes = plt.subplots(1, len(data), figsize=(6 * len(data), 4))
for i, yy in enumerate(y_unique):
    ax = axes[i]
    cor = correlation_matrices[yy]
    nedis.visualization.visualize_feature_clusters(cor, ax=ax, heatmap_kwargs=dict(cmap="vlag", center=0, vmin=-1, vmax=1))

In [ ]:
fig, axes = plt.subplots(1, len(y_unique), figsize=(4 * 2 * len(y_unique), 4 * 2))
for i, yy in enumerate(y_unique):
    ax = axes[i]
    ax.axis("off")
    ax.set_title(yy)
    plot_cordis_cluster(None, coordinates[y_unique[0]], correlation_matrices[yy], ax=ax)

# Custom clusters

In [ ]:
from nedis.cordis.clustering import (
    ReferenceFeatureLabelClusteringStep, ListClusteringStep, init_cluster)
from nedis.cordis.optimization import (
    GreedyRefinementOptimizationStep, ReferenceScoreOptimizationStep)

In [ ]:
from nedis.cordis.utils import calculate_disruption_values_for_cluster_from_data

In [ ]:
clustering_step = ListClusteringStep([
    init_cluster(reference_label=0, reference_shape=X.shape[1], rows=np.arange(0,5)),
    init_cluster(reference_label=4, reference_shape=X.shape[1], rows=np.arange(5,15)),
    init_cluster(reference_label=0, reference_shape=X.shape[1], rows=np.arange(15,25)),
])

# clustering_step.fit(
#     X, None, reference_label="test", reference_mask=np.ones(X.shape[0], dtype=bool));

In [ ]:
# clustering_step = ReferenceFeatureLabelClusteringStep(labels)

# clustering_step.fit_reference(
#     X, None, reference_label="test", reference_mask=np.ones(X.shape[0], dtype=bool));

In [ ]:
optimization_step = GreedyRefinementOptimizationStep(
    separation_score="spearman",
#     separation_score_comparison='all',
#     refinement_mode="rows-and-columns",
#     correlation_function="spearman",
#     disruption_metric="direction",
#     disruption_robustness='loo',
#     disruption_aggregation='mean',
#     max_runs=-1
)

# ALTERNATIVE: use this to prevent optimization
# optimization_step = ReferenceScoreOptimizationStep(
#     separation_score="spearman",
# )

t = CorrelationDisruptionTransformer(
    clustering_step=clustering_step,
    cluster_optimization_step=optimization_step,
    filter_coverage_threshold=0.5, 
    separation_score_threshold=("auto", 1)
)
t.fit(X, y, subset_masks="y")

In [ ]:
t.filter_reset()
t.filter_clusters_by_threshold(separation_score_threshold=None)
t.filter_clusters_by_overlap(filter_coverage_threshold=None)

clusters = list(sorted([c for c in t.clusters_ if c["selected"]], key=lambda c: c["reference_score"]))
len(clusters)

In [ ]:
if "disruption_matrices_dict" not in globals():
    disruption_matrices_dict = {}
    for yy in np.unique(y):
        print("Calculating disruption values for reference:", yy)
        disruption_matrices_dict[yy] = calculate_correlation_disruption_matrix_cv(X, idx_ref=(y == yy))
disruption_matrices = disruption_matrices_dict[0]

In [ ]:
for i_cluster, cluster in enumerate(reversed(clusters)):
    
    fig, axes = plt.subplots(1, len(y_unique) + 1, figsize=(4 * 1 * (len(y_unique) + 1), 4 * 1), dpi=300)
 
    values, _ = calculate_disruption_values_for_cluster(
        cluster, disruption_matrices_dict[cluster["reference_label"]], disruption_aggregation="mean")
   
#     ax = axes[0]
#     sns.kdeplot(x=values, hue=y, ax=ax, palette=sns.color_palette(n_colors=len(y_unique)))
#     ax.set_title(f"{scipy.stats.spearmanr(values, y)[0]:.04f}")
      
    r, p = scipy.stats.spearmanr(values, y)
    
    ax = axes[0]
    x_rank = scipy.stats.rankdata(y, method="dense")
    sns.lineplot(x=x_rank, y=values, hue=entities, color="blue", alpha=0.1, ax=ax)
    sns.lineplot(x=x_rank, y=values, ax=ax)
    ax.set(
        xlabel="timepoints",
        xticks=np.unique(x_rank),
        xticklabels= y_unique,
        ylabel="disruption",
        title=f"r={r:.02f}"
    )
    ax.get_legend().remove()
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    
    for i, yy in enumerate(y_unique):
        ax = axes[i + 1]
        ax.axis("off")
        plot_cluster(
            cluster, 
            coordinates[y_unique[0]], 
            correlation_matrices[yy], 
            correlation_threshold=0, 
            verbose=0, 
            ax=ax)
    fig.suptitle(f"Cluster: Reference data={cluster['reference_label']}, Id={cluster['id']}")
    
    # fig.savefig(output_dir / f"modules_{i_cluster}.png", bbox_inches="tight")
    # fig.savefig(output(output_dir / f"modules_{i_cluster}.png", bbox_inches="tight")
    # fig.savefig(output_d_dir / f"modules_{i_cluster}.pdf", bbox_inches="tight")
    
    plt.show()
#     break